# AC-MOT v10_p4 — FP16 Fair Benchmark + Deployment Validation

v10_p3 remains untouched. Dataset and recorded videos are staged to `/content` once per Colab runtime and reused locally afterward.

In [ ]:
# CELL 1 — GPU + INSTALL + DRIVE MOUNT
!nvidia-smi
!pip install -q ultralytics==8.3.200 motmetrics opencv-python-headless pandas numpy tqdm scipy lap pyyaml
!pip install -q git+https://github.com/JonathonLuiten/TrackEval.git
import os, sys, torch, ultralytics
from google.colab import drive
if not os.path.isdir('/content/drive/MyDrive'):
    drive.mount('/content/drive', force_remount=True)
print('Python:', sys.version.split()[0])
print('Torch:', torch.__version__)
print('Ultralytics:', ultralytics.__version__)
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
assert torch.cuda.is_available(), 'CUDA is not available'
assert 'T4' in torch.cuda.get_device_name(0), f'Expected Tesla T4, found {torch.cuda.get_device_name(0)}'
print('PRE-FLIGHT OK')


In [ ]:
# CELL 2 — FRESH PUBLIC CLONE (FAST, NO TOKEN)
import pathlib, shutil, subprocess
REPO='/content/ACMOT-Codex-V10Style-Portable'
REPO_URL='https://github.com/AhmedCode110/ACMOT-Codex-V10Style-Portable.git'
if pathlib.Path(REPO).exists(): shutil.rmtree(REPO)
subprocess.run(['git','clone','--depth','1',REPO_URL,REPO],check=True)
commit=subprocess.check_output(['git','-C',REPO,'rev-parse','HEAD'],text=True).strip()
print('Repo ready:', REPO)
print('Commit:', commit)
assert pathlib.Path(REPO,'fair_benchmark_v10_p4.py').exists()
assert pathlib.Path(REPO,'drone_runtime_v10_p4.py').exists()


In [ ]:
# CELL 3 — SMART DATASET STAGING: COPY ONLY ONCE PER RUNTIME
from pathlib import Path
import json, shutil, time, pandas as pd
from tqdm.auto import tqdm
from google.colab import drive
DRIVE_DATASET=Path('/content/drive/MyDrive/visdrone/VisDrone_Zips/VisDrone2019-MOT-test-dev/VisDrone2019-MOT-test-dev')
LOCAL=Path('/content/visdrone_v10_p4_local/VisDrone2019-MOT-test-dev')
READY=LOCAL/'.acmot_local_ready.json'
EXPECTED_SEQS=17
def local_is_ready():
    if not READY.exists(): return False
    try:
        meta=json.loads(READY.read_text())
        seq_dir=LOCAL/'sequences'; ann_dir=LOCAL/'annotations'
        seqs=sorted([p.name for p in seq_dir.iterdir() if p.is_dir()])
        if len(seqs)!=EXPECTED_SEQS: return False
        total=0
        for s in seqs:
            frames=list((seq_dir/s).glob('*.jpg'))
            if not (ann_dir/f'{s}.txt').exists(): return False
            total += len(frames)
        return total==meta.get('total_frames') and meta.get('sequences')==EXPECTED_SEQS
    except Exception:
        return False
if local_is_ready():
    print('LOCAL DATASET ALREADY READY — SKIP STAGING')
    print('Using:', LOCAL)
else:
    if not Path('/content/drive/MyDrive').exists(): drive.mount('/content/drive', force_remount=True)
    SEQ=DRIVE_DATASET/'sequences'; ANN=DRIVE_DATASET/'annotations'
    assert SEQ.exists() and ANN.exists(), 'Drive dataset path is unavailable. Remount Drive and rerun Cell 3.'
    seqs=sorted([p.name for p in SEQ.iterdir() if p.is_dir()])
    assert len(seqs)==EXPECTED_SEQS, f'Expected 17 sequences, found {len(seqs)}'
    manifest=[]
    for s in seqs:
        frames=sorted((SEQ/s).glob('*.jpg'))
        gt=pd.read_csv(ANN/f'{s}.txt',header=None)
        gt_max=int(gt.iloc[:,0].max())
        assert len(frames)==gt_max, f'Mismatch {s}: frames={len(frames)}, gt_max={gt_max}'
        manifest.append((s,len(frames)))
    if LOCAL.exists(): shutil.rmtree(LOCAL)
    (LOCAL/'sequences').mkdir(parents=True); (LOCAL/'annotations').mkdir(parents=True)
    total=sum(n for _,n in manifest)+len(manifest)
    p=tqdm(total=total,desc='ONE-TIME Drive -> /content staging',dynamic_ncols=True)
    for s,n in manifest:
        dst=LOCAL/'sequences'/s; dst.mkdir()
        for fp in sorted((SEQ/s).glob('*.jpg')):
            shutil.copy2(fp,dst/fp.name); p.update(1)
        shutil.copy2(ANN/f'{s}.txt',LOCAL/'annotations'/f'{s}.txt'); p.update(1)
    p.close()
    READY.write_text(json.dumps({'sequences':len(manifest),'total_frames':sum(n for _,n in manifest)},indent=2))
    assert local_is_ready(), 'Local staging verification failed'
    print('ONE-TIME STAGING COMPLETE')
    print('Future reruns of Cell 3 in THIS runtime will skip copying.')
print('Local dataset:', LOCAL)


In [ ]:
# CELL 4 — FP16 FAIR BENCHMARK + OFFICIAL TRACKEVAL
from datetime import datetime
from pathlib import Path
import subprocess, sys, shutil, torch
from ultralytics import YOLO
assert (LOCAL/'.acmot_local_ready.json').exists(), 'Run Cell 3 first'
WEIGHTS=Path('/content/yolov8n.pt')
if not WEIGHTS.exists():
    print('Downloading official YOLOv8n weights...')
    m=YOLO('yolov8n.pt')
    candidate=Path('yolov8n.pt').resolve()
    if candidate != WEIGHTS and candidate.exists(): shutil.copy2(candidate, WEIGHTS)
    del m
assert WEIGHTS.exists()
OUT=Path('/content/drive/MyDrive/VisDrone_Results/ACMOT_CODEX_V10STYLE/V10_P4_FP16_FAIR') / ('codex_v10_p4_fp16_'+datetime.now().strftime('%Y%m%d_%H%M%S'))
cmd=[sys.executable, f'{REPO}/fair_benchmark_v10_p4.py','--dataset',str(LOCAL),'--weights',str(WEIGHTS),'--output',str(OUT),'--run-trackeval']
print('Running:', ' '.join(cmd))
proc=subprocess.run(cmd,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT)
print(proc.stdout)
if proc.returncode!=0:
    print('===== LAST 120 LINES OF REAL ERROR =====')
    print('\n'.join((proc.stdout or '').splitlines()[-120:]))
    raise RuntimeError(f'v10_p4 benchmark failed with exit code {proc.returncode}')
print('FINAL RESULT FOLDER:', OUT)


In [ ]:
# CELL 5 — ONE-TIME VIDEO STAGING TO LOCAL /content
from pathlib import Path
import shutil
# Change only this path to your recorded drone video on Drive:
VIDEO_DRIVE=Path('/content/drive/MyDrive/your_flight_video.mp4')
LOCAL_VIDEO_DIR=Path('/content/acmot_local_videos'); LOCAL_VIDEO_DIR.mkdir(exist_ok=True)
LOCAL_VIDEO=LOCAL_VIDEO_DIR/VIDEO_DRIVE.name
if LOCAL_VIDEO.exists() and LOCAL_VIDEO.stat().st_size>0:
    print('LOCAL VIDEO ALREADY READY — SKIP COPY')
else:
    assert VIDEO_DRIVE.exists(), f'Video not found: {VIDEO_DRIVE}'
    print('Copying video ONCE from Drive to local disk...')
    shutil.copy2(VIDEO_DRIVE,LOCAL_VIDEO)
print('Use this local video for all runs:', LOCAL_VIDEO)


In [ ]:
# CELL 6 — RUN AC-MOT ON THE LOCAL VIDEO
from pathlib import Path
DEPLOY_OUT=Path('/content/acmot_drone_runs')
DEPLOY_OUT.mkdir(exist_ok=True)
assert LOCAL_VIDEO.exists(), 'Run Cell 5 first'
!python {REPO}/drone_runtime_v10_p4.py --source "{LOCAL_VIDEO}" --weights /content/yolov8n.pt --output-dir "{DEPLOY_OUT}" --save-video
print('Local deployment output:', DEPLOY_OUT)
